# LifeMate — notebook đo thời gian suy luận (ZAC 2026)

Chạy **Run All**. Notebook dùng đúng các hàm của `/code/predict.py` (cùng lối với `predict.sh`), xử lý **lần lượt từng mẫu**:
đồng hồ bắt đầu trước khi đọc audio → một `ToolEnv` cho mẫu → ASR → bộ lọc app + LLM gọi tool → `tool_calls`, `state_hash`, `time` (ms).

Kết quả: `/result/jupyter_submission.jsonl` (cùng định dạng `submission.jsonl`, có trường `time`). Không cần mạng.

## Bước 1 — Đặt seed cố định

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())          # notebook nằm cùng thư mục với predict.py (/code)
sys.path.insert(1, "/code")
import predict as P

SEED = P.seed_everything(0)              # seed của đội: 0 (Python, NumPy, PyTorch; ASR và vLLM cũng chạy seed 0, giải mã tất định)
print("seed =", SEED)

## Bước 2 — Nạp mô hình và tài nguyên

ASR (PhoWhisper, tiến trình riêng) → vLLM phục vụ LLM gọi tool → tokenizer, catalog, bộ lọc app. Không dùng API ngoài.

In [ ]:
rt = P.load_models()                     # in "load model: ... ms"

## Bước 3 — Đọc các test case từ `/data`

In [ ]:
import json
from pathlib import Path

test_cases = P.read_cases(rt)            # /data/<split>/<split>.jsonl ; split tự tìm theo <split>_tools_catalog.json
print(rt.split, "-", len(test_cases), "mẫu")

## Bước 4 — Dự đoán lần lượt từng mẫu, ghi kết quả và thời gian

In [ ]:
import time

out_path = Path(P.CFG["OUT"]).parent / "jupyter_submission.jsonl"      # /result/jupyter_submission.jsonl
out_path.parent.mkdir(parents=True, exist_ok=True)

all_result, t_all = [], time.perf_counter()
with open(out_path, "w", encoding="utf-8") as f:
    for n, item in enumerate(test_cases, 1):                           # lần lượt từng mẫu
        # P.infer: t1 = perf_counter() trước khi đọc audio → ToolEnv (đúng một cái) → ASR → LLM ⇄ env.call → hash
        tool_calls, state_hash, ms = P.infer(rt, item)
        r = {"id": item["id"], "tool_calls": tool_calls, "state_hash": state_hash, "time": ms}
        all_result.append(r)
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
        f.flush()
        if n % 20 == 0 or n == len(test_cases):
            print(f"{n}/{len(test_cases)} mẫu | mẫu vừa xong {ms} ms", flush=True)

P.report(rt, [r["time"] for r in all_result], time.perf_counter() - t_all, str(out_path))

## Dọn dẹp — tắt ASR và vLLM, nhả GPU

In [ ]:
P.shutdown(rt)
print("xong:", out_path)